In [ ]:
%cd ../../

In [ ]:
import datetime
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import polars as pl
import darts
from darts import TimeSeries
from darts.timeseries import concatenate
from darts.dataprocessing.transformers import Scaler
from pytorch_lightning.callbacks import TQDMProgressBar
from lightning.pytorch.loggers import TensorBoardLogger
from sklearn.metrics import mean_absolute_percentage_error
from darts.models import CatBoostModel, TFTModel, TransformerModel, TSMixerModel, RNNModel, XGBModel, LightGBMModel, NBEATSModel
from sklearn.preprocessing import MinMaxScaler

In [ ]:
EPS = 1e-6

In [ ]:
plt.style.use('seaborn-v0_8')
plt.rcParams.update({'font.size': 8})

# Read and process data

In [ ]:
path = "data/processed/pos.xlsx"
pos = pl.read_excel(path)

pos.head()

In [ ]:
path = "data/processed/waste.parquet"

waste = pl.read_parquet(path)
waste.head()

In [ ]:
path = "data/processed/dim_restaurants.xlsx"
dim_restaurants = pl.read_excel(path)

dim_restaurants.head()

In [ ]:
path = "data/processed/dim_meals.parquet"
dim_meals = pl.read_parquet(path)

dim_meals.head()

## Process

In [ ]:
waste_daily = (
    waste
    .drop('id', 'src')

    # .join(
    #     dim_restaurants.select('restaurant_id', 'restaurant_short'),
    #     left_on='restaurant',
    #     right_on='restaurant_id',
    #     how='left'
    # )
    # .drop('restaurant')
    # .rename({'restaurant_short': 'restaurant'})   


    # Add small amount for training stability
    .with_columns(
        pl.col('waste') + EPS
    )


    # Pivot
    .pivot(index='date', on='restaurant', values='waste')
    .sort('date')

)


# Make new dataframe only containing entries of weekdays
waste_daily = (
    pl.DataFrame({
        'date': pl.Series(pd.date_range(waste['date'].min(), waste['date'].max(), freq='B')).cast(pl.Date)
    })

    .join(waste_daily, on='date', how='left')

    .fill_null(EPS)
)


waste_daily.head()

In [ ]:
restaurant = "1"

series = (
    TimeSeries
    .from_dataframe(
        waste_daily.to_pandas(),
        time_col='date',
        value_cols=[
            restaurant
            # 'che',
            # 'exa',
            # 'phy',
            # 'vik'
        ],
        fillna_value=EPS,
        freq='B',
    )
)

series.plot()

## Tailor other dimensions

In [ ]:
path = "data/processed/dim_exams_uhelsinki.xlsx"
dim_exam = (
     pl.read_excel(path)
    .filter(pl.col('date').dt.weekday() <= 5)
)


series_exam = (
    TimeSeries
    .from_dataframe(
        df=dim_exam.to_pandas(),
        time_col='date',
        freq='b',
        fill_missing_dates = False,
        value_cols='is_exam'
    )
)
series_exam.plot()

In [ ]:
path = "data/processed/dim_holidays_uhelsinki.xlsx"
dim_holiday = (
    pl.read_excel(path)
    .filter(pl.col('date').dt.weekday() <= 5)
)


series_holiday = (
    TimeSeries
    .from_dataframe(
        df=dim_holiday.to_pandas(),
        time_col='date',
        freq='b',
        fill_missing_dates = True,
        value_cols='is_holiday'
    )
    # .astype(np.float32)
)
series_holiday.plot()


In [ ]:
dim_meal_types_count = (
    pos
    .with_columns(
        pl.col('datetime').dt.date().alias('date')
    )
    .select('restaurant', 'date', 'meal_id')
    .unique()

    .join(
        dim_meals.select('id', 'meal_type'),
        left_on='meal_id',
        right_on='id',
        how='left'
    )
    .filter(pl.col('meal_type') <= 5)
    .group_by('date', 'restaurant', 'meal_type')
    .len('count')

    .join(
        dim_restaurants.select('restaurant_id', 'restaurant_short'),
        left_on='restaurant',
        right_on='restaurant_id',
        how='left'
    )
    .drop('restaurant')
    .rename({'restaurant_short': 'restaurant'})   

    .pivot(index=['date', 'restaurant'], on='meal_type', values='count')
    .fill_null(0)


    .filter(pl.col('restaurant') == pl.lit(restaurant))
    .filter(pl.col('date').dt.weekday() <= 5)
)

series_meal_types = (
    TimeSeries
    .from_dataframe(
        df=dim_meal_types_count.to_pandas(),
        time_col='date',
        freq='b',
        fill_missing_dates = True,
        fillna_value=0,
        value_cols=[
            '1',
            '2',
            '3',
            '4',
            '5'
        ]
    )
    # .astype(np.float32)
)
series_meal_types.plot()

# Build forecasting model

In [ ]:
CUTOFF_DATE = pd.to_datetime("2025-01-01")
min_time = max(
    series_exam.time_index[0],
    series_holiday.time_index[0],
    # series_meal_types.time_index[0],
)
max_time = min(
    series_exam.time_index[-1],
    series_holiday.time_index[-1],
    # series_meal_types.time_index[-1],
)

series_train, series_test = series[min_time:max_time].astype(np.float32).split_before(CUTOFF_DATE)

series_cov = concatenate([
    series_exam[min_time:max_time],
    series_holiday[min_time:max_time],
    # series_meal_types[min_time:max_time],
], axis=1).astype(np.float32)

series_cov.plot()

In [ ]:
# transformer = Diff(1, dropna=True)
# series_transformed = transformer.fit_transform(series)
# series_train_diff = series_transformed.drop_after(CUTOFF_DATE)


series_train_transformed = series_train
# series_train_transformed = series_train_diff

transformer_target = Scaler(MinMaxScaler(feature_range=(-1, 1)))
# transformer_target = BoxCox(lmbda=0.)
series_train_transformed = transformer_target.fit_transform(series_train_transformed)

transformer_cov = Scaler(MinMaxScaler(feature_range=(-1, 1)))
series_cov_transformed = transformer_cov.fit_transform(series_cov)


fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111)

series_train_transformed.plot(ax=ax, label='train')
series_cov_transformed.plot(ax=ax)

## Models

In [ ]:
version = datetime.datetime.now().strftime("%m-%d_%H-%M-%S")
model_name = "rnn"

input_chunk_length = 3
output_chunk_length = 60
num_epochs = 5


# Define params
add_encoders = {
    "datetime_attribute": {
        "future": ["dayofweek", 'day', 'month'],
        "past": ["dayofweek", 'day', 'month']
    },
    'cyclic': {
        'past': ["dayofweek", 'day', 'month'],
        'future': ["dayofweek", 'day', 'month']
    },
}

params_ml = {
    "lags": input_chunk_length,
    "lags_future_covariates": [0],
    "output_chunk_length": output_chunk_length,
    "add_encoders": {**add_encoders}
}
params_dl = {
    "input_chunk_length": input_chunk_length,
    "output_chunk_length": output_chunk_length,
    "add_encoders": {**add_encoders},  
    "n_epochs": num_epochs,
    "pl_trainer_kwargs": {
        "callbacks": [TQDMProgressBar(refresh_rate=4)],
        "logger": [
            TensorBoardLogger("logs/tensorboard", name=f"{model_name}-{restaurant}", version=version, default_hp_metric=False)
        ],
        'precision': "32-true",
    },
    "optimizer_kwargs": {
        'lr': 5e-4
    }
}

# Define models
match model_name:
    # =================================================
    # ML models
    # =================================================
    case "catboost":
        model = CatBoostModel(**params_ml)
        model.fit(
            series_train_transformed,
            future_covariates=series_cov_transformed,
        )
    case "xgboost":
        model = XGBModel(**params_ml)
        model.fit(
            series_train_transformed,
            future_covariates=series_cov_transformed,
        )
    case "lightbgm":
        model = LightGBMModel(**params_ml)
        model.fit(
            series_train_transformed,
            future_covariates=series_cov_transformed,
        )

    # =================================================
    # DL models
    # =================================================
    case 'rnn':
        params_dl['model'] = 'LSTM'

        model = RNNModel(**params_dl)
        model.fit(
            series_train_transformed,
            future_covariates=series_cov_transformed,
        )
    case "tsmixer":
        model = TSMixerModel(**params_dl)
        model.fit(
            series_train_transformed,
            future_covariates=series_cov_transformed,
        )
    case "transformer":
        model = TransformerModel(**params_dl)
        model.fit(
            series_train_transformed,
            past_covariates=series_cov_transformed.split_before(CUTOFF_DATE)[0],
        )
    case "tft":
        params_dl["categorical_embedding_sizes"] = {
            "holiday": (2, 2),
            "exam": (2, 2),
            "restaurant": (4, 4)
        }

        model = TFTModel(**params_dl)
        model.fit(
            series_train_transformed,
            future_covariates=series_cov_transformed,
        )
    case "n-beats":
        model = NBEATSModel(**params_dl)
        model.fit(
            series_train_transformed,
            past_covariates=series_cov_transformed.split_before(CUTOFF_DATE)[0],
        )
    
    case _:
        raise NotImplementedError()

## Test

In [ ]:
match model_name:
    # =================================================
    # ML models
    # =================================================
    case "catboost":
        series_pred = model.predict(
            n=len(series_test),
            future_covariates=series_cov_transformed
        )
    case "xgboost":
        series_pred = model.predict(
            n=len(series_test),
            future_covariates=series_cov_transformed
        )
    case "lightbgm":
        series_pred = model.predict(
            n=len(series_test),
            future_covariates=series_cov_transformed
        )

    # =================================================
    # DL models
    # =================================================
    case "rnn":
        series_pred = model.predict(
            n=len(series_test),
            future_covariates=series_cov_transformed
        )
    case "tsmixer":
        series_pred = model.predict(
            n=len(series_test),
            future_covariates=series_cov_transformed
        )
    case "transformer":
        series_pred = model.predict(
            n=len(series_test),
            past_covariates=series_cov_transformed
        )
    case "transformer":
        series_pred = model.predict(
            n=len(series_test),
            past_covariates=series_cov_transformed
        )
    case "tft":
        series_pred = model.predict(
            n=len(series_test),
            future_covariates=series_cov_transformed
        )
    case "n-beats":
        series_pred = model.predict(
            n=len(series_test),
            past_covariates=series_cov_transformed
        )

    case _:
        raise NotImplementedError()
    

series_pred = transformer_target.inverse_transform(series_pred)
# series_pred = transformer.inverse_transform(series_train_diff.concatenate(series_pred)).drop_before(CUTOFF_DATE)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111)
series_pred.plot(ax=ax, label='pred')
series_test.plot(ax=ax, label='gt')
# rmse_val = mape(series_test[comp], series_pred[comp])

df = pd.concat(
    [
        series_test.to_dataframe().rename(columns={restaurant: "gt"}),
        series_pred.to_dataframe().rename(columns={restaurant: "pred"})
    ],
    axis=1
)
df = df[df['gt'] > EPS]
mape_val = mean_absolute_percentage_error(df['gt'], df['pred'])

ax.set_title(f"{restaurant} | mape = {mape_val*100:.4}%")

# Export to production model

Use entire `series` and `series_cov` for prediction

In [ ]:
tag = "May_26"
restaurant = "4"

path_dir = Path(f"trained_models/whole_restaurant_waste/May_26_{restaurant}")

path_dir.mkdir(exist_ok=True, parents=True)

path_model = path_dir / "model.pt"
path_scaler_tgt = path_dir / "scaler_tgt.gz"
path_scaler_cov = path_dir / "scaler_cov.gz"

In [ ]:
series = (
    TimeSeries
    .from_dataframe(
        waste_daily.to_pandas(),
        time_col='date',
        value_cols=restaurant,
        fillna_value=EPS,
        freq='B',
    )
)
series_cov = concatenate([
    series_exam,
    series_holiday,
], axis=1).astype(np.float32)

In [ ]:
transformer_target = Scaler(MinMaxScaler(feature_range=(-1, 1)))
transformer_cov = Scaler(MinMaxScaler(feature_range=(-1, 1)))

series_transformed = transformer_target.fit_transform(series)
series_cov_transformed = transformer_cov.fit_transform(series_cov)

In [ ]:
assert isinstance(series_cov_transformed, TimeSeries)


input_chunk_length = 3
output_chunk_length = 1
num_epochs = 200


# Define params
add_encoders = {
    "datetime_attribute": {
        "future": ["dayofweek", 'day', 'month'],
    },
    'cyclic': {
        'past': ["dayofweek", 'day', 'month'],
    },
}
params_dl = {
    "input_chunk_length": input_chunk_length,
    "output_chunk_length": output_chunk_length,
    "add_encoders": {**add_encoders},  
    "n_epochs": num_epochs,
    "pl_trainer_kwargs": {
        'precision': "32-true",
        'logger': False,
        'enable_progress_bar': True,
        'enable_model_summary': False,
    },
    "optimizer_kwargs": {
        'lr': 5e-4
    }
}

model = TransformerModel(**params_dl)
model.fit(series_transformed, past_covariates=series_cov_transformed)

### Verify model training with backtest

In [ ]:
preds_raw = model.historical_forecasts(series_transformed, start=pd.to_datetime('2025-01-01'), retrain=False)
preds = transformer_target.inverse_transform(preds_raw)
assert isinstance(preds, TimeSeries)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111)
preds.plot(ax=ax, label='pred')
series[pd.to_datetime('2025-01-01'):].plot(ax=ax, label='gt')

## Save model and scaler

In [ ]:
joblib.dump(transformer_target, path_scaler_tgt)
joblib.dump(transformer_cov, path_scaler_cov)
model.save(path_model.as_posix(), clean=False)

## Test loading model and making prediction

In [ ]:
transformer_target = joblib.load(path_scaler_tgt)
transformer_cov = joblib.load(path_scaler_cov)

model = (
    TransformerModel(**params_dl)
    .load(path_model.as_posix(), map_location='cpu', pl_trainer_kwargs={'accelerator': 'cpu'})
)
# model = model

In [ ]:
preds_raw = model.predict(20)
preds = transformer_target.inverse_transform(preds_raw)

preds.to_dataframe()